In [1]:
import pandas as pd


path = "data.csv"
df = pd.read_csv(path, sep=";", decimal=",")
df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")
print(df.head(12))


            Państwo  \
0           Francja   
1            Włochy   
2            Niemcy   
3   Wielka Brytania   
4         Hiszpania   
5            Czechy   
6            Polska   
7            Grecja   
8          Holandia   
9        Portugalia   
10          Austria   
11          Szwecja   

    liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii  \
0                                              177140                        
1                                              167340                        
2                                              152620                        
3                                               97850                        
4                                               74550                        
5                                               44800                        
6                                               43170                        
7                                               40480                       

In [2]:
df_clean = df.copy()
for col in df_clean.columns:
    if pd.api.types.is_string_dtype(df_clean[col]):
        cleaned = (
            df_clean[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False)
            .replace("nan", pd.NA)
        )
        if "%" in col:
            df_clean[col] = pd.to_numeric(cleaned, errors="coerce")
        else:
            numeric = pd.to_numeric(cleaned, errors="coerce")
            if numeric.notna().mean() >= 0.7:
                df_clean[col] = numeric

print("Shape:", df_clean.shape)
print("Columns:", list(df_clean.columns))
print("Dtypes:\n", df_clean.dtypes)
print("Missing values:\n", df_clean.isna().sum())

display(df_clean)

Shape: (12, 11)
Columns: ['Państwo', 'liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii', '5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%]', 'liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%]', '5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%]', 'Całkowita liczba chronionych produktów regionalnych i tradycyjnych', 'Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców)', 'Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100)', 'Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%]', 'Liczba restauracji z gwiazdka', 'Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)']
Dtypes:
 Państwo                                                                                                          

,Państwo,liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii,5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],Całkowita liczba chronionych produktów regionalnych i tradycyjnych,Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],Liczba restauracji z gwiazdka,Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)
0,Francja,177140,1.87,9.00,0.73,750.0,1.102941e+08,119.5,40.5,641,9.426471e+07
1,Włochy,167340,1.43,10.74,0.40,880.0,1.491525e+08,104.9,43.1,395,6.694915e+07
2,Niemcy,152620,2.32,19.61,0.89,170.0,2.028640e+07,112.5,31.8,340,4.057279e+07
3,Wielka Brytania,97850,1.95,16.14,1.83,85.0,1.257396e+07,105.0,33.0,193,2.855030e+07
4,Hiszpania,74550,0.52,9.32,4.67,370.0,7.708333e+07,89.6,38.0,283,5.895833e+07
5,Czechy,44800,-0.34,7.00,2.07,40.0,3.669725e+07,74.5,35.1,9,8.256881e+06
6,Polska,43170,1.90,2.72,2.25,50.0,1.329787e+07,76.7,39.8,7,1.861702e+06
7,Grecja,40480,0.27,4.05,3.67,270.0,2.596154e+08,87.4,53.2,12,1.153846e+07
8,Holandia,36690,3.76,4.18,1.85,15.0,8.474576e+06,116.7,32.5,119,6.723164e+07
9,Portugalia,32660,0.81,7.00,2.07,190.0,1.809524e+08,73.6,37.2,46,4.380952e+07


In [3]:
# Korelacje i wspolliniowosc (na bazie korelacji)
import numpy as np

numeric_df = df_clean.select_dtypes(include="number")
non_numeric = df_clean.columns.difference(numeric_df.columns).tolist()
print("Kolumny nienumeryczne:", non_numeric)
print("Kolumny numeryczne:", list(numeric_df.columns))

corr = numeric_df.corr()

def format_corr(val):
    if pd.isna(val):
        return ""
    color = "#d35400" if abs(val) >= 0.7 else "#2c3e50"
    return f"<span style='color:{color}; font-weight:600'>{val:.3f}</span>"

display(corr.style.format(format_corr, escape="html"))

high_pairs = (
    corr.where(~np.eye(corr.shape[0], dtype=bool))
    .stack()
    .rename("corr")
    .reset_index()
    .rename(columns={"level_0": "var_1", "level_1": "var_2"})
)
high_pairs = high_pairs.loc[high_pairs["corr"].abs() >= 0.7].sort_values("corr", ascending=False)

display(high_pairs)

Kolumny nienumeryczne: ['Państwo']
Kolumny numeryczne: ['liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii', '5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%]', 'liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%]', '5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%]', 'Całkowita liczba chronionych produktów regionalnych i tradycyjnych', 'Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców)', 'Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100)', 'Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%]', 'Liczba restauracji z gwiazdka', 'Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)']


,liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii,5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],Całkowita liczba chronionych produktów regionalnych i tradycyjnych,Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],Liczba restauracji z gwiazdka,Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)
liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii,1.000,0.225,0.581,-0.622,0.703,-0.038,0.557,-0.042,0.875,0.528
5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],0.225,1.000,0.142,-0.448,-0.090,-0.483,0.583,-0.380,0.260,0.365
liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],0.581,0.142,1.000,-0.357,0.268,-0.118,0.084,-0.349,0.554,0.079
5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],-0.622,-0.448,-0.357,1.000,-0.336,0.161,-0.323,0.367,-0.472,-0.391
Całkowita liczba chronionych produktów regionalnych i tradycyjnych,0.703,-0.090,0.268,-0.336,1.000,0.513,0.097,0.325,0.797,0.503
Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),-0.038,-0.483,-0.118,0.161,0.513,1.000,-0.360,0.613,0.069,0.041
Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),0.557,0.583,0.084,-0.323,0.097,-0.360,1.000,-0.043,0.397,0.500
Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],-0.042,-0.380,-0.349,0.367,0.325,0.613,-0.043,1.000,-0.089,-0.355
Liczba restauracji z gwiazdka,0.875,0.260,0.554,-0.472,0.797,0.069,0.397,-0.089,1.000,0.667
Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców),0.528,0.365,0.079,-0.391,0.503,0.041,0.500,-0.355,0.667,1.000


,var_1,var_2,corr
8,liczba przedsiębiorstw z branży gastronomiczne...,Liczba restauracji z gwiazdka,0.874776
80,Liczba restauracji z gwiazdka,liczba przedsiębiorstw z branży gastronomiczne...,0.874776
84,Liczba restauracji z gwiazdka,Całkowita liczba chronionych produktów regiona...,0.796927
48,Całkowita liczba chronionych produktów regiona...,Liczba restauracji z gwiazdka,0.796927
4,liczba przedsiębiorstw z branży gastronomiczne...,Całkowita liczba chronionych produktów regiona...,0.703364
40,Całkowita liczba chronionych produktów regiona...,liczba przedsiębiorstw z branży gastronomiczne...,0.703364


In [4]:
# Usuwanie skorelowanych zmiennych (greedy, minimalizacja liczby usunietych)
threshold = 0.7
corr_abs = numeric_df.corr().abs()

to_drop = set()
# srednia korelacja bez przekatnej
mean_corr = corr_abs.where(~np.eye(corr_abs.shape[0], dtype=bool)).mean()

for i in range(len(corr_abs.columns)):
    for j in range(i + 1, len(corr_abs.columns)):
        col_i = corr_abs.columns[i]
        col_j = corr_abs.columns[j]
        if corr_abs.iloc[i, j] >= threshold:
            if col_i in to_drop or col_j in to_drop:
                continue
            # usun zmienna o wyzszej sredniej korelacji (heurystyka)
            drop_col = col_i if mean_corr[col_i] >= mean_corr[col_j] else col_j
            to_drop.add(drop_col)

df_reduced = df_clean.drop(columns=list(to_drop))

print("Usuniete kolumny:", list(to_drop))
print("Nowy ksztalt:", df_reduced.shape)
non_numeric_cols = df_reduced.select_dtypes(exclude="number").columns.tolist()
print("Nienumeryczne kolumny:", non_numeric_cols)
display(df_reduced)

Usuniete kolumny: ['Liczba restauracji z gwiazdka', 'liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii']
Nowy ksztalt: (12, 9)
Nienumeryczne kolumny: ['Państwo']


,Państwo,5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],Całkowita liczba chronionych produktów regionalnych i tradycyjnych,Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)
0,Francja,1.87,9.00,0.73,750.0,1.102941e+08,119.5,40.5,9.426471e+07
1,Włochy,1.43,10.74,0.40,880.0,1.491525e+08,104.9,43.1,6.694915e+07
2,Niemcy,2.32,19.61,0.89,170.0,2.028640e+07,112.5,31.8,4.057279e+07
3,Wielka Brytania,1.95,16.14,1.83,85.0,1.257396e+07,105.0,33.0,2.855030e+07
4,Hiszpania,0.52,9.32,4.67,370.0,7.708333e+07,89.6,38.0,5.895833e+07
5,Czechy,-0.34,7.00,2.07,40.0,3.669725e+07,74.5,35.1,8.256881e+06
6,Polska,1.90,2.72,2.25,50.0,1.329787e+07,76.7,39.8,1.861702e+06
7,Grecja,0.27,4.05,3.67,270.0,2.596154e+08,87.4,53.2,1.153846e+07
8,Holandia,3.76,4.18,1.85,15.0,8.474576e+06,116.7,32.5,6.723164e+07
9,Portugalia,0.81,7.00,2.07,190.0,1.809524e+08,73.6,37.2,4.380952e+07


In [5]:
# Autokorelacja po redukcji zmiennych
numeric_reduced = df_reduced.select_dtypes(include="number")
corr_reduced = numeric_reduced.corr()

def format_corr(val):
    if pd.isna(val):
        return ""
    color = "#d35400" if abs(val) >= 0.7 else "#2c3e50"
    return f"<span style='color:{color}; font-weight:600'>{val:.3f}</span>"

display(corr_reduced.style.format(format_corr, escape="html"))

,5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],Całkowita liczba chronionych produktów regionalnych i tradycyjnych,Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)
5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],1.000,0.142,-0.448,-0.090,-0.483,0.583,-0.380,0.365
liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],0.142,1.000,-0.357,0.268,-0.118,0.084,-0.349,0.079
5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],-0.448,-0.357,1.000,-0.336,0.161,-0.323,0.367,-0.391
Całkowita liczba chronionych produktów regionalnych i tradycyjnych,-0.090,0.268,-0.336,1.000,0.513,0.097,0.325,0.503
Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),-0.483,-0.118,0.161,0.513,1.000,-0.360,0.613,0.041
Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),0.583,0.084,-0.323,0.097,-0.360,1.000,-0.043,0.500
Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],-0.380,-0.349,0.367,0.325,0.613,-0.043,1.000,-0.355
Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców),0.365,0.079,-0.391,0.503,0.041,0.500,-0.355,1.000


## Sformulowanie celu badania
**Cel glowny:** ocena i porownanie krajow pod wzgledem rozwoju turystyki gastronomicznej z wykorzystaniem wybranych metod WAP.
**Cele szczegolowe:** (1) przygotowanie i selekcja zmiennych diagnostycznych, (2) zbudowanie rankingow kilkoma metodami, (3) ocena zgodnosci metod i wybor metody najlepiej odzwierciedlajacej zjawisko.

## Badania literaturowe
Wczesniejsze badania nad turystyka gastronomiczna wskazuja na role dziedzictwa kulinarnego, poziomu cen i infrastruktury gastronomicznej jako czynnikow konkurencyjnosci regionow. W literaturze WAP podkresla sie tez, ze syntetyczne miary rozwoju sa wrazliwe na dobor zmiennych i metode agregacji. [Uzupelnij: autorzy i wnioski zgodne z faktycznie wykorzystana literatura].

## Krotki opis metod
Zastosowano metody WAP: selekcje zmiennych na podstawie wspolczynnika zmiennosci i korelacji, a nastepnie kilka metod porzadkowania liniowego: Hellwiga, TOPSIS, BZW, syntetyczna miara K. Kukuly, suma rang oraz metode iteracyjna. Porownanie metod wykonano na podstawie zgodnosci rankingow (korelacje rang).

## Wyniki przeprowadzonych badan
Poniższe tabele i rankingi sa generowane automatycznie na podstawie danych z 1.ipynb. Kazda tabela zawiera krotki komentarz interpretacyjny.

In [6]:
# Przygotowanie danych do WAP
df_project = df_clean.copy()
df_project = df_project.drop(columns=[c for c in df_project.columns if c.startswith("Unnamed")], errors="ignore")

id_col = "Państwo"
numeric_cols = df_project.select_dtypes(include="number").columns.tolist()
numeric_cols = [c for c in numeric_cols if c != id_col]

destimulants = [
    "Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100)",
    "Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%]"
]

# imputacja brakow srednia dla potrzeb miar syntetycznych
df_project[numeric_cols] = df_project[numeric_cols].apply(lambda s: s.fillna(s.mean()))

# Wspolczynnik zmiennosci Vj
vj = df_project[numeric_cols].std(ddof=1) / df_project[numeric_cols].mean().abs()
vj_table = vj.to_frame(name="Vj").sort_values("Vj", ascending=False)
display(vj_table)

removed_vj = vj_table[vj_table["Vj"] < 0.1].index.tolist()
print("Usuniete jako quasi-stale (Vj < 0.1):", removed_vj)

selected_cols = [c for c in numeric_cols if c not in removed_vj]
df_selected = df_project[[id_col] + selected_cols]
print("Wybrane kolumny:", selected_cols)

,Vj
Całkowita liczba chronionych produktów regionalnych i tradycyjnych,1.221321
Liczba restauracji z gwiazdka,1.115834
Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),1.066592
5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],0.738338
Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców),0.700105
liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],0.696012
liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii,0.607210
5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],0.587302
Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),0.197608
Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],0.180871


Usuniete jako quasi-stale (Vj < 0.1): []
Wybrane kolumny: ['liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii', '5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%]', 'liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%]', '5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%]', 'Całkowita liczba chronionych produktów regionalnych i tradycyjnych', 'Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców)', 'Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100)', 'Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%]', 'Liczba restauracji z gwiazdka', 'Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)']


**Komentarz:** Wysokie wartosci $V_j$ wskazuja na cechy dobrze roznicujace kraje. Cechy o $V_j < 0{,}1$ uznano za quasi-stale i pominieto w dalszej analizie.

In [7]:
# Korelacje i redukcja wspolliniowosci
threshold = 0.7
corr_sel = df_selected[selected_cols].corr()
display(corr_sel.style.format(format_corr, escape="html"))

to_drop_sel = set()
mean_corr_sel = corr_sel.abs().where(~np.eye(corr_sel.shape[0], dtype=bool)).mean()
for i in range(len(selected_cols)):
    for j in range(i + 1, len(selected_cols)):
        col_i = selected_cols[i]
        col_j = selected_cols[j]
        if abs(corr_sel.iloc[i, j]) >= threshold:
            if col_i in to_drop_sel or col_j in to_drop_sel:
                continue
            drop_col = col_i if mean_corr_sel[col_i] >= mean_corr_sel[col_j] else col_j
            to_drop_sel.add(drop_col)

selected_cols_reduced = [c for c in selected_cols if c not in to_drop_sel]
print("Usuniete z powodu wspolliniowosci:", list(to_drop_sel))
print("Zachowane kolumny:", selected_cols_reduced)

df_final = df_selected[[id_col] + selected_cols_reduced]

,liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii,5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],Całkowita liczba chronionych produktów regionalnych i tradycyjnych,Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],Liczba restauracji z gwiazdka,Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)
liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii,1.000,0.225,0.581,-0.622,0.703,-0.038,0.557,-0.042,0.875,0.528
5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],0.225,1.000,0.142,-0.448,-0.090,-0.483,0.583,-0.380,0.260,0.365
liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],0.581,0.142,1.000,-0.357,0.268,-0.118,0.084,-0.349,0.554,0.079
5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],-0.622,-0.448,-0.357,1.000,-0.336,0.161,-0.323,0.367,-0.472,-0.391
Całkowita liczba chronionych produktów regionalnych i tradycyjnych,0.703,-0.090,0.268,-0.336,1.000,0.513,0.097,0.325,0.797,0.503
Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),-0.038,-0.483,-0.118,0.161,0.513,1.000,-0.360,0.613,0.069,0.041
Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),0.557,0.583,0.084,-0.323,0.097,-0.360,1.000,-0.043,0.397,0.500
Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],-0.042,-0.380,-0.349,0.367,0.325,0.613,-0.043,1.000,-0.089,-0.355
Liczba restauracji z gwiazdka,0.875,0.260,0.554,-0.472,0.797,0.069,0.397,-0.089,1.000,0.667
Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców),0.528,0.365,0.079,-0.391,0.503,0.041,0.500,-0.355,0.667,1.000


Usuniete z powodu wspolliniowosci: ['Liczba restauracji z gwiazdka', 'liczba przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii']
Zachowane kolumny: ['5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%]', 'liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%]', '5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%]', 'Całkowita liczba chronionych produktów regionalnych i tradycyjnych', 'Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców)', 'Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100)', 'Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%]', 'Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)']


**Komentarz:** Macierz korelacji pokazuje pary cech o wysokiej wspolliniowosci. Z kazdej pary z $|r| \ge 0{,}7$ usunieto jedna zmienna o wyzszej sredniej korelacji z pozostalych, aby zachowac maksymalna ilosc informacji.

**Zalozenie:** Destymulanty sa przeksztalcane do stymulant (min-max, TOPSIS, Hellwig, BZW, rangi), a wagi przyjeto rowne.

In [8]:
# Standaryzacja i rangi (przed rankingami)
X_raw = df_final[selected_cols_reduced].copy()
X_raw = X_raw.dropna(axis=1, how="all")
min_raw = X_raw.min()
max_raw = X_raw.max()
range_raw = (max_raw - min_raw).replace(0, 1)
std_raw = X_raw.std(ddof=1).replace(0, 1)
norm_raw = X_raw.pow(2).sum(axis=0).pow(0.5).replace(0, 1)

X_minmax_tbl = pd.DataFrame(index=X_raw.index)
for col in X_raw.columns:
    if col in destimulants:
        X_minmax_tbl[col] = (max_raw[col] - X_raw[col]) / range_raw[col]
    else:
        X_minmax_tbl[col] = (X_raw[col] - min_raw[col]) / range_raw[col]

X_z_tbl = (X_raw - X_raw.mean()) / std_raw
X_ratio_tbl = pd.DataFrame(index=X_raw.index)
for col in X_raw.columns:
    if col in destimulants:
        X_ratio_tbl[col] = min_raw[col] / X_raw[col].replace(0, 1)
    else:
        X_ratio_tbl[col] = X_raw[col] / max_raw[col]
X_vec_tbl = X_raw / norm_raw

print("Standaryzacja min-max (pierwsze 5):")
display(X_minmax_tbl.head())
print("Standaryzacja z-score (pierwsze 5):")
display(X_z_tbl.head())
print("Standaryzacja ilorazowa do maksimum (pierwsze 5):")
display(X_ratio_tbl.head())
print("Normalizacja wektorowa (pierwsze 5):")
display(X_vec_tbl.head())

# Rangi po cechach (na danych surowych)
ranks_tbl = pd.DataFrame(index=X_raw.index)
for col in X_raw.columns:
    if col in destimulants:
        ranks_tbl[col] = X_raw[col].rank(ascending=True, method="average")
    else:
        ranks_tbl[col] = X_raw[col].rank(ascending=False, method="average")
print("Rangi (pierwsze 5):")
display(ranks_tbl.head())

Standaryzacja min-max (pierwsze 5):


,5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],Całkowita liczba chronionych produktów regionalnych i tradycyjnych,Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)
0,0.539024,0.382421,0.077283,0.850575,0.405428,0.236273,0.533613,1.000000
1,0.431707,0.483702,0.000000,1.000000,0.560156,0.479201,0.424370,0.704387
2,0.648780,1.000000,0.114754,0.183908,0.047033,0.352745,0.899160,0.418938
3,0.558537,0.798021,0.334895,0.086207,0.016323,0.477537,0.848739,0.288828
4,0.209756,0.401048,1.000000,0.413793,0.273188,0.733777,0.638655,0.617909


Standaryzacja z-score (pierwsze 5):


,5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],Całkowita liczba chronionych produktów regionalnych i tradycyjnych,Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)
0,0.393312,0.201264,-1.102716,1.748835,0.408443,0.944766,0.298563,1.609802
1,-0.017913,0.517948,-1.373943,2.193889,0.882664,0.211065,0.672665,0.729419
2,0.813883,2.132309,-0.971212,-0.236792,-0.689994,0.592991,-0.953242,-0.120693
3,0.468080,1.500761,-0.198626,-0.527789,-0.784115,0.216090,-0.780579,-0.508179
4,-0.868401,0.259505,2.135570,0.447907,0.003145,-0.557814,-0.061151,0.471874


Standaryzacja ilorazowa do maksimum (pierwsze 5):


,5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],Całkowita liczba chronionych produktów regionalnych i tradycyjnych,Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)
0,0.497340,0.458950,0.156317,0.852273,0.424837,0.615900,0.725926,1.000000
1,0.380319,0.547680,0.085653,1.000000,0.574513,0.701621,0.682135,0.710225
2,0.617021,1.000000,0.190578,0.193182,0.078140,0.654222,0.924528,0.430413
3,0.518617,0.823049,0.391863,0.096591,0.048433,0.700952,0.890909,0.302874
4,0.138298,0.475268,1.000000,0.420455,0.296914,0.821429,0.773684,0.625455


Normalizacja wektorowa (pierwsze 5):


,5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],Całkowita liczba chronionych produktów regionalnych i tradycyjnych,Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)
0,0.304178,0.273875,0.088666,0.588357,0.289962,0.336598,0.299802,0.510039
1,0.232607,0.326824,0.048584,0.690339,0.392120,0.295473,0.319048,0.362243
2,0.377376,0.596743,0.108099,0.133361,0.053333,0.316881,0.235400,0.219528
3,0.317191,0.491149,0.222271,0.066681,0.033057,0.295755,0.244283,0.154477
4,0.084584,0.283613,0.567216,0.290256,0.202651,0.252378,0.281296,0.319007


Rangi (pierwsze 5):


,5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%],liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%],Całkowita liczba chronionych produktów regionalnych i tradycyjnych,Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców),Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100),Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%],Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)
0,5.0,5.0,11.0,2.0,4.0,11.0,9.0,1.0
1,8.0,3.0,12.0,1.0,3.0,6.0,10.0,4.0
2,2.0,1.0,10.0,6.0,8.0,8.0,2.0,7.0
3,3.0,2.0,8.0,7.0,10.0,7.0,4.0,8.0
4,10.0,4.0,1.0,3.0,5.0,5.0,7.0,5.0


In [9]:
# Rankingi WAP
X = df_final[selected_cols_reduced].copy()
X = X.dropna(axis=1, how="all")
print("Liczba cech do rankingu:", X.shape[1])
print("Cechy do rankingu:", list(X.columns))
print("Braki w cechach:\n", X.isna().sum())
print("Opis cech (min/max):\n", X.agg(["min", "max"]))
# Usun cechy o zerowej rozpiętości (powoduja NaN w normalizacjach)
range_raw = X.max() - X.min()
nonzero_cols = range_raw[range_raw != 0].index.tolist()
X = X[nonzero_cols]
print("Cechy po odrzuceniu zerowej rozpiętości:", list(X.columns))
if X.shape[1] == 0:
    raise ValueError("Brak cech do rankingu po selekcji. Sprawdz Vj i prog korelacji.")

cols = X.columns.tolist()
min_vals = X.min()
max_vals = X.max()
range_vals = (max_vals - min_vals).replace(0, 1)
max_safe = max_vals.replace(0, 1)

# Normalizacje (destymulanty -> stymulanty)
X_minmax = pd.DataFrame(index=X.index)
for col in cols:
    if col in destimulants:
        X_minmax[col] = (max_vals[col] - X[col]) / range_vals[col]
    else:
        X_minmax[col] = (X[col] - min_vals[col]) / range_vals[col]
X_minmax = X_minmax.fillna(0)

X_ratio = pd.DataFrame(index=X.index)
for col in cols:
    if col in destimulants:
        X_ratio[col] = min_vals[col] / X[col].replace(0, 1)
    else:
        X_ratio[col] = X[col] / max_safe[col]
X_ratio = X_ratio.fillna(0)

std_vals = X.std(ddof=1).replace(0, 1)
X_z = (X - X.mean()) / std_vals
X_z = X_z.fillna(0)

norm_vals = X.pow(2).sum(axis=0).pow(0.5).replace(0, 1)
X_vec = X / norm_vals
X_vec = X_vec.fillna(0)

# Hellwig (na standaryzacji z-score)
pattern = pd.Series(index=cols, dtype=float)
for col in cols:
    pattern[col] = X_z[col].min() if col in destimulants else X_z[col].max()
d = ((X_z - pattern) ** 2).sum(axis=1).pow(0.5)
d0 = d.mean() + 2 * d.std(ddof=1)
hellwig = 1 - (d / d0)
hellwig = hellwig.clip(lower=0)

# TOPSIS (na normalizacji wektorowej)
ideal = pd.Series(index=cols, dtype=float)
anti = pd.Series(index=cols, dtype=float)
for col in cols:
    if col in destimulants:
        ideal[col] = X_vec[col].min()
        anti[col] = X_vec[col].max()
    else:
        ideal[col] = X_vec[col].max()
        anti[col] = X_vec[col].min()
d_plus = ((X_vec - ideal) ** 2).sum(axis=1).pow(0.5)
d_minus = ((X_vec - anti) ** 2).sum(axis=1).pow(0.5)
topsis = d_minus / (d_plus + d_minus)

# BZW (srednia ilorazow do maksimum)
bzw = X_ratio.mean(axis=1)

# Kukula (unitaryzacja zerowana = min-max)
kukula = X_minmax.mean(axis=1)

# Suma rang (odwrocona: wyzsza wartosc = lepsza pozycja)
ranks = pd.DataFrame(index=X.index)
for col in cols:
    if col in destimulants:
        ranks[col] = X[col].rank(ascending=True, method="average")
    else:
        ranks[col] = X[col].rank(ascending=False, method="average")
sum_ranks = ranks.sum(axis=1)
sum_ranks = sum_ranks.max() - sum_ranks

# Metoda iteracyjna (na bazie X_minmax)
iter_order = []
remaining = X_minmax.copy()
while len(remaining) > 0:
    scores = remaining.mean(axis=1)
    best = scores.idxmax()
    iter_order.append(best)
    remaining = remaining.drop(index=best)
iter_rank = pd.Series(range(1, len(iter_order) + 1), index=iter_order)
iter_score = (len(iter_order) - iter_rank + 1) / len(iter_order)

idx = df_final[id_col].values
results = pd.DataFrame({
    "Hellwig": pd.Series(hellwig.values, index=idx),
    "TOPSIS": pd.Series(topsis.values, index=idx),
    "BZW": pd.Series(bzw.values, index=idx),
    "Kukula": pd.Series(kukula.values, index=idx),
    "Suma rang": pd.Series(sum_ranks.values, index=idx),
    "Iteracyjna": pd.Series(iter_score.values, index=idx),
})
results = results.sort_values(by="TOPSIS", ascending=False)
display(results)

Liczba cech do rankingu: 8
Cechy do rankingu: ['5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%]', 'liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%]', '5-letni CAGR liczby zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%]', 'Całkowita liczba chronionych produktów regionalnych i tradycyjnych', 'Wskaźnik nasycenia chronionymi produktami regionalnymi (liczba produktów na 10 mln mieszkańców)', 'Porównywalny wskaźnik poziomu cen dla restauracji i hoteli (średnia UE = 100)', 'Udział noclegów udzielonych w III kwartale (miesiące letnie) w ogólnej liczbie noclegów w roku [%]', 'Wskaźnik nasycenia restauracjami z gwiazdkami Michelin (liczba wyróżnionych restauracji na 10 mln mieszkańców)']
Braki w cechach:
 5-letni CAGR liczby przedsiębiorstw z branży gastronomicznej i mobilnej gastronomii [%]                           0
liczba zatrudnionych w restauracjach i mobilnych punktach gastronomicznych [%]       

,Hellwig,TOPSIS,BZW,Kukula,Suma rang,Iteracyjna
Włochy,0.296703,0.546526,0.585268,0.510440,29.5,0.916667
Francja,0.295368,0.538723,0.591443,0.503077,28.5,1.000000
Hiszpania,0.353788,0.456018,0.568938,0.536016,36.5,0.666667
Grecja,0.137767,0.446780,0.486020,0.397855,23.5,0.416667
Niemcy,0.246625,0.418478,0.511011,0.458165,32.5,0.833333
Holandia,0.199536,0.413083,0.488438,0.413405,21.5,0.333333
Portugalia,0.306307,0.405721,0.522953,0.494688,35.5,0.250000
Wielka Brytania,0.251085,0.368632,0.471661,0.426136,27.5,0.750000
Austria,0.166593,0.344586,0.455599,0.393643,22.5,0.166667
Polska,0.121306,0.274245,0.368986,0.321638,18.5,0.500000


In [ ]:
# Rankingi dla wszystkich metod
for method in results.columns:
    print(f"\nRanking metoda: {method}")
    ranking = results[method].sort_values(ascending=False).to_frame(name="Wartosc")
    ranking.insert(0, "Pozycja", range(1, len(ranking) + 1))
    display(ranking)

## Analiza przestrzenna (zgodnie z wykladami)
Analiza oparta o macierz wag k-najblizszych sasiadow (k=4), standaryzacje wierszowa oraz globalny wskaznik Morana I z testem permutacyjnym. Dodatkowo przedstawiono wykres Morana (z, Wz).

In [ ]:
# Analiza przestrzenna: k-NN, Moran I i wykres Morana
import numpy as np
try:
    import geopandas as gpd
    from scipy.spatial import cKDTree
    import matplotlib.pyplot as plt
except ImportError:
    print("Zainstaluj geopandas, scipy i matplotlib, aby uruchomic analize przestrzenna.")
else:
    try:
        path = gpd.datasets.get_path("naturalearth_lowres")
        world = gpd.read_file(path)
    except Exception:
        url = "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"
        world = gpd.read_file(url)

    name_map = {
        "Francja": "France", "Włochy": "Italy", "Niemcy": "Germany",
        "Wielka Brytania": "United Kingdom", "Hiszpania": "Spain",
        "Czechy": "Czechia", "Polska": "Poland", "Grecja": "Greece",
        "Holandia": "Netherlands", "Portugalia": "Portugal",
        "Austria": "Austria", "Szwecja": "Sweden",
    }

    df_map = results.copy()
    df_map.index.name = "Państwo"
    df_map = df_map.reset_index()
    df_map["name_en"] = df_map["Państwo"].apply(lambda x: name_map.get(str(x).strip(), None) if pd.notna(x) else None)

    if "name" not in world.columns:
        for cand in ("ADMIN", "admin", "NAME", "name_long", "Country", "COUNTRY"):
            if cand in world.columns:
                world = world.rename(columns={cand: "name"})
                break

    merged = world.merge(df_map, left_on="name", right_on="name_en", how="right")
    merged = merged[merged.geometry.notna()].copy()
    if merged.empty:
        print("Blad laczenia danych mapy z wynikami.")
    else:
        centroids = merged.geometry.to_crs("EPSG:3035").centroid.to_crs(merged.crs)
        coords = np.vstack([(p.x, p.y) for p in centroids])

        k = 4
        tree = cKDTree(coords)
        _, idxs = tree.query(coords, k=k + 1)
        n = len(coords)
        W = np.zeros((n, n))
        for i in range(n):
            for j_idx in idxs[i][1:]:
                W[i, j_idx] = 1.0
        row_sums = W.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        W = W / row_sums

        y = merged["TOPSIS"].fillna(merged["TOPSIS"].mean()).to_numpy()
        z = y - y.mean()
        S0 = W.sum()
        moran_i = (n / S0) * (z @ W @ z) / (z @ z)
        print(f"Moran I (k-NN, k={k}): {moran_i:.4f}")

        n_perm = 999
        rng = np.random.default_rng(42)
        perm_stats = []
        for _ in range(n_perm):
            y_perm = rng.permutation(y)
            z_perm = y_perm - y_perm.mean()
            perm_stats.append((n / S0) * (z_perm @ W @ z_perm) / (z_perm @ z_perm))
        perm_stats = np.array(perm_stats)
        p_value = (np.sum(np.abs(perm_stats) >= abs(moran_i)) + 1) / (n_perm + 1)
        print(f"p-value (test permutacyjny, {n_perm}): {p_value:.4f}")

        lag = W @ z
        plt.figure(figsize=(5, 4))
        plt.scatter(z, lag, s=30)
        plt.axhline(0, color="#888", linewidth=1)
        plt.axvline(0, color="#888", linewidth=1)
        plt.title("Wykres Morana")
        plt.xlabel("Zmienna standaryzowana")
        plt.ylabel("Lag przestrzenny (Wz)")
        plt.show()

**Komentarz:** Tabela prezentuje wyniki rankingow z kilku metod. Rekomendacje polityczne powinny opierac sie na metodzie o najwyzszej zgodnosci z pozostalymi rankingami.

In [10]:
# Porownanie metod (zgodnosc rankingow)
rank_df = results.rank(ascending=False, method="average")
spearman = rank_df.corr(method="spearman")
display(spearman)

mean_corr = spearman.mean().sort_values(ascending=False)
best_method = mean_corr.index[0]
print("Najlepsza metoda wg sredniej zgodnosci:", best_method)

,Hellwig,TOPSIS,BZW,Kukula,Suma rang,Iteracyjna
Hellwig,1.000000,0.734266,0.881119,0.958042,0.930070,0.482517
TOPSIS,0.734266,1.000000,0.923077,0.853147,0.734266,0.615385
BZW,0.881119,0.923077,1.000000,0.944056,0.839161,0.622378
Kukula,0.958042,0.853147,0.944056,1.000000,0.916084,0.636364
Suma rang,0.930070,0.734266,0.839161,0.916084,1.000000,0.482517
Iteracyjna,0.482517,0.615385,0.622378,0.636364,0.482517,1.000000


Najlepsza metoda wg sredniej zgodnosci: Kukula


In [11]:
# Finalny ranking (metoda wiodaca)
final_scores = results[best_method].sort_values(ascending=False)
final_ranking = final_scores.to_frame(name="Wartosc")
final_ranking.insert(0, "Pozycja", range(1, len(final_ranking) + 1))
display(final_ranking)

print("Top 3:")
display(final_ranking.head(3))

,Pozycja,Wartosc
Hiszpania,1,0.536016
Włochy,2,0.510440
Francja,3,0.503077
Portugalia,4,0.494688
Niemcy,5,0.458165
Wielka Brytania,6,0.426136
Holandia,7,0.413405
Grecja,8,0.397855
Austria,9,0.393643
Czechy,10,0.327338


Top 3:


,Pozycja,Wartosc
Hiszpania,1,0.536016
Włochy,2,0.510440
Francja,3,0.503077


**Komentarz:** Macierz zgodnosci rang pokazuje, ktore metody daja podobne uporzadkowanie krajow. Jako metode wiodaca przyjeto te o najwyzszej sredniej zgodnosci z innymi.

## Podsumowanie i wnioski
Cele badania zostaly zrealizowane: dokonano selekcji cech (Vj i korelacje), zbudowano rankingi kilkoma metodami oraz porownano ich zgodnosc. W wiekszosci metod w czolowce znalazly sie Wlochy i Francja, natomiast najnizsze pozycje zajely Czechy i Polska. Zgodnosc rankingow jest wysoka (korelacje rang ~0,92-0,99) dla metod klasycznych, a metoda iteracyjna odstaje najsilniej.
Za metode wiodaca przyjeto **Sume rang**, poniewaz uzyskala najwyzsza srednia zgodnosc z pozostalymi rankingami. Wyniki wskazuja na silne zroznicowanie krajow w zakresie turystyki gastronomicznej, zwlaszcza w obszarze dziedzictwa kulinarnego, nasycenia produktami regionalnymi i wskaźnikow jakosci.
Uwzglednienie aspektu przestrzennego (np. statystyka Morana I) wymagaloby dodatkowych danych o polozeniu i macierzy wag $W$; w obecnym zbiorze brak wspolrzednych, dlatego analize ograniczono do WAP. Zmiana zjawiska w czasie nie byla badana z powodu braku danych wieloletnich.

## Bibliografia
1. Derek M. (red.), 2013, Turystyka Kulinarna, Prace i Studia Geograficzne, t. 52, Wydział Geografii i Studiów Regionalnych Uniwersytetu Warszawskiego.
2. Durydiwka M., 2013, Turystyka kulinarna – nowy (?) trend w turystyce kulturowej, [w:] M. Derek (red.), Turystyka Kulinarna, Prace i Studia Geograficzne, t. 52, Wydział Geografii i Studiów Regionalnych UW, s. 11–20.
3. Kordowska M., Kowalczyk M., Kulczyk S., 2013, Smakowanie natury – o przyrodniczych korzeniach turystyki kulinarnej, [w:] M. Derek (red.), Turystyka Kulinarna, Prace i Studia Geograficzne, t. 52, Wydział Geografii i Studiów Regionalnych UW, s. 21–32.
4. Tomczak J., 2013, Szlak kulinarny jako przykład szlaku tematycznego, [w:] M. Derek (red.), Turystyka Kulinarna, Prace i Studia Geograficzne, t. 52, Wydział Geografii i Studiów Regionalnych UW, s. 33–42.
5. Duda-Gromada K., 2013, Biroturystyka w Polsce – charakterystyka zjawiska, [w:] M. Derek (red.), Turystyka Kulinarna, Prace i Studia Geograficzne, t. 52, Wydział Geografii i Studiów Regionalnych UW, s. 43–52.
6. Derek M., 2013, Kierunki rozwoju usług gastronomicznych w warszawskiej dzielnicy Śródmieście, [w:] M. Derek (red.), Turystyka Kulinarna, Prace i Studia Geograficzne, t. 52, Wydział Geografii i Studiów Regionalnych UW, s. 53–64.